# Import Libraries and packages

In [2]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(
    { "figure.figsize": (8, 5) },
    style='ticks',
    color_codes=True,
    font_scale=1
)
%config InlineBackend.figure_format = 'retina'

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

!pip install category_encoders -q
!pip install xgboost

# ML utilities
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn import metrics

# Models
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans

# Encoding and metrics
from category_encoders import TargetEncoder
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, classification_report

# Hyperparameter tuning
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

# Loading the dataset

In [4]:
ecom_data = pd.read_csv("../data/Dataset_ecommerce.csv")

# Initial inspection of the dataset

In [6]:
ecom_data.head()

,InvoiceNo,InvoiceDate,CustomerID,StockCode,Description,Quantity,UnitPrice,Country
0,536365,2010-12-01 08:26:00,17850.0,SC1734,Electronics,65,10.23,Egypt
1,536365,2010-12-01 08:26:00,17850.0,SC2088,Furniture,95,19.61,Mali
2,536365,2010-12-01 08:26:00,17850.0,SC3463,Books,78,61.49,Mali
3,536365,2010-12-01 08:26:00,17850.0,SC6228,Toys,15,24.73,South Africa
4,536365,2010-12-01 08:26:00,17850.0,SC2149,Toys,50,38.83,Rwanda


In [7]:
ecom_data.sample()

,InvoiceNo,InvoiceDate,CustomerID,StockCode,Description,Quantity,UnitPrice,Country
518741,580115,2011-12-01 16:22:00,NaN,SC8873,Books,93,25.71,Togo


In [8]:
ecom_data.shape

(541909, 8)

In [9]:
ecom_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   InvoiceDate  541909 non-null  object 
 2   CustomerID   406829 non-null  float64
 3   StockCode    541909 non-null  object 
 4   Description  541909 non-null  object 
 5   Quantity     541909 non-null  int64  
 6   UnitPrice    541909 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


### Observation: Sampled Dataset contains 541909 rows and 8 columns in which most of them are strings.

In [11]:
ecom_data.describe(include='all')

,InvoiceNo,InvoiceDate,CustomerID,StockCode,Description,Quantity,UnitPrice,Country
count,541909,541909,406829.000000,541909,541909,541909.000000,541909.000000,541909
unique,25900,23260,NaN,9000,10,NaN,NaN,28
top,573585,2011-10-31 14:41:00,NaN,SC2014,Sports Equipment,NaN,NaN,Cote d'Ivoire
freq,1114,1114,NaN,96,54765,NaN,NaN,19651
mean,NaN,NaN,15287.690570,NaN,NaN,50.534748,50.476354,NaN
std,NaN,NaN,1713.600303,NaN,NaN,28.849367,28.564775,NaN
min,NaN,NaN,12346.000000,NaN,NaN,1.000000,1.000000,NaN
25%,NaN,NaN,13953.000000,NaN,NaN,26.000000,25.750000,NaN
50%,NaN,NaN,15152.000000,NaN,NaN,51.000000,50.430000,NaN
75%,NaN,NaN,16791.000000,NaN,NaN,76.000000,75.180000,NaN


In [12]:
# Checking Unique values in the columns
print("Unique Customers:", ecom_data['CustomerID'].nunique())
print("Unique Products:", ecom_data['StockCode'].nunique())
print("Countries:", ecom_data['Country'].nunique())

Unique Customers: 4372
Unique Products: 9000
Countries: 28


In [21]:
ecom_data.isnull().sum()

InvoiceNo           0
InvoiceDate         0
CustomerID     135080
StockCode           0
Description         0
Quantity            0
UnitPrice           0
Country             0
dtype: int64

### Observation - The CustomerID column have 135080 null values (missing data).

# Splitting the features to qualitative and quantitative features

In [25]:
# Categorising quantitative features (numerical columns)
numerical_features = ecom_data.select_dtypes(include=['float64', 'int64']).columns.tolist()
# Categorising qualitative features (categorical columns)
categorical_features = ecom_data.select_dtypes(include=['object', 'bool']).columns.tolist()

In [27]:
numerical_features

['CustomerID', 'Quantity', 'UnitPrice']

In [29]:
categorical_features

['InvoiceNo', 'InvoiceDate', 'StockCode', 'Description', 'Country']

# Data Cleaning & Preprocessing

## Handling missing values, duplicate values and formatting the CustomerID and InvoiceDate 

In [33]:
# the percentage of missing values in each columns
missing_pct = round(ecom_data.isna().sum()/len(ecom_data) * 100, 1)
print(missing_pct)

InvoiceNo       0.0
InvoiceDate     0.0
CustomerID     24.9
StockCode       0.0
Description     0.0
Quantity        0.0
UnitPrice       0.0
Country         0.0
dtype: float64


### Observation - ~25% of the CustomerID values are missing which is a significant portion of the dataset. However, since the goal of the project is churn prediction, it is better to drop rows with missing CustomerID in order to have a clean base of identifiable customers for all the cohort and ML modeling tasks.

In [36]:
# Drop rows with missing CustomerID
ecom_data = ecom_data.dropna(subset=['CustomerID'])

In [38]:
# Confirm if the missing CustomerID are dropped
print(f"Remaining rows: {ecom_data.shape[0]}")
print(f"Missing CustomerID after drop: {ecom_data['CustomerID'].isna().sum()}")

Remaining rows: 406829
Missing CustomerID after drop: 0


In [40]:
# Check number of duplicates before dropping
duplicate_count = ecom_data.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count}")

Number of duplicate rows: 0


In [42]:
# Convert the CustomerID from float to integer
ecom_data = ecom_data.dropna(subset=['CustomerID'])
ecom_data['CustomerID'] = ecom_data['CustomerID'].astype(int)

In [44]:
# Convert the date format from string to datetime
ecom_data['InvoiceDate'] = pd.to_datetime(ecom_data['InvoiceDate'])

In [46]:
# check the range of dates
print(ecom_data['InvoiceDate'].min(), "to", ecom_data['InvoiceDate'].max())

2010-12-01 08:26:00 to 2011-12-09 12:50:00


# Feature engineering based on domain knowledge

In [49]:
# Create TotalPrice feature
ecom_data['TotalPrice'] = ecom_data['Quantity'] * ecom_data['UnitPrice']

In [51]:
# Extract features such as Invoice Year, Invoice Month and Weekday
ecom_data['InvoiceYear'] = ecom_data['InvoiceDate'].dt.year
ecom_data['InvoiceMonth'] = ecom_data['InvoiceDate'].dt.to_period('M')
ecom_data['Weekday'] = ecom_data['InvoiceDate'].dt.day_name()

In [53]:
ecom_data.head(1)

,InvoiceNo,InvoiceDate,CustomerID,StockCode,Description,Quantity,UnitPrice,Country,TotalPrice,InvoiceYear,InvoiceMonth,Weekday
0,536365,2010-12-01 08:26:00,17850,SC1734,Electronics,65,10.23,Egypt,664.95,2010,2010-12,Wednesday
